# 01 — McNemar's Test for Paired Classification Accuracy

Use McNemar's test when two classifiers are evaluated on the **same test examples** and each example is either correct or incorrect for each model.

It is more appropriate than simply comparing two accuracies because it uses the paired structure of the evaluation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make src/ importable when running from the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from significance_utils import *

DATA_DIR = PROJECT_ROOT / 'data' / 'dummy'
ALPHA = 0.05

## Student input

You can change `ALPHA` if you want a stricter or looser significance threshold.

In [ ]:
ALPHA = 0.05
DATA_FILE = DATA_DIR / 'classification_predictions.csv'

In [ ]:
df = pd.read_csv(DATA_FILE)
df.head()

## Compute accuracies

In [ ]:
df['a_correct'] = df['model_a_pred'] == df['gold_label']
df['b_correct'] = df['model_b_pred'] == df['gold_label']

acc_a = df['a_correct'].mean()
acc_b = df['b_correct'].mean()

print(f'Model A accuracy: {acc_a:.3f}')
print(f'Model B accuracy: {acc_b:.3f}')
print(f'Difference B - A: {acc_b - acc_a:.3f}')

## Run McNemar's test

McNemar's test focuses only on the examples where the two models disagree:

- only Model A correct
- only Model B correct

If Model B is truly better, we expect more examples in the second category.

In [ ]:
result = mcnemar_exact_test(
    y_true=df['gold_label'],
    pred_a=df['model_a_pred'],
    pred_b=df['model_b_pred'],
)

pd.Series(result)

In [ ]:
print(interpret_p_value(result['p_value'], alpha=ALPHA))

## Visualize the contingency table

In [ ]:
contingency = pd.DataFrame(
    [
        [result['both_correct'], result['only_b_correct']],
        [result['only_a_correct'], result['both_wrong']],
    ],
    index=['Model A correct', 'Model A incorrect'],
    columns=['Model B correct', 'Model B incorrect'],
)
contingency

In [ ]:
ax = contingency.plot(kind='bar')
ax.set_ylabel('Number of examples')
ax.set_title('Paired correctness outcomes')
plt.xticks(rotation=0)
plt.show()

## Exercise questions

1. Why does McNemar's test ignore the cases where both models are correct or both are wrong?
2. What happens to the p-value if you manually change some predictions so that Model B has more unique correct cases?
3. Why would an unpaired test be inappropriate here?